# 1.6 章节实践：实现行 Softmax

本节是第 1 章的综合实践。前面几节分别学习了逐元素计算、矩阵乘法、规约计算和形状操作。本节将其中的逐元素、规约和广播组合起来，实现一个按最后一维计算的 Softmax 算子。

本节不引入新的复杂 API，而是把已经学过的内容组合起来：`amax/sum` 负责规约，`sub/exp/div` 负责逐元素计算，`keepdim=True` 负责保留广播所需的维度，最后用 PyTorch 的 `torch.softmax` 建立验证闭环。


---
## 前置说明

本节在第一个代码单元格中集中放置环境准备代码（import、设备选择、`RUN_MODE` 等）。后续每个可运行模块在定义 kernel 前只调用 `reset_pypto_notebook_state()` 重置编译状态，不再重复完整的环境初始化。

环境准备的具体含义参见 `01.01_chapter_intro.ipynb` §1。


## 直观理解：Softmax 是把分数变成比例

Softmax 可以先理解为“把一组分数变成一组比例”。例如一行里有几个分数：

```python
[2.0, 1.0, 0.1]
```

Softmax 会把它们变成类似“概率”的数字：

- 每个数字都大于 0。
- 加起来约等于 1。
- 原来分数越大的位置，转换后通常也越大。

所以在 Attention 里，Softmax 常用来表示“应该更关注哪些位置”。


## 2. Softmax 的数学含义

Softmax 常用于把一组分数转换为概率分布。对于一行输入 `x`，Softmax 可以写成：

```python
softmax(x_i) = exp(x_i) / sum(exp(x_i))
```

输出通常具有两个特点：

1. 每个元素大于 0。
2. 每一行元素之和接近 1。

在注意力机制中，Softmax 常用于把 `q @ k.T` 得到的分数转换成权重。


## 3. 为什么要减去 row_max

直接计算 `exp(x)` 可能遇到数值溢出。例如某些输入值很大时，指数结果会非常大。为了提升数值稳定性，常见做法是先减去每一行最大值：

```python
row_max = max(x)
shifted = x - row_max
```

这个操作不会改变 Softmax 的数学结果，因为同一行所有元素都减去了同一个常数，最终归一化后的比例保持不变。

在 PyPTO 中，我们使用：

```python
row_max = pypto.amax(x, dim=-1, keepdim=True)
```

这里 `keepdim=True` 很重要，它让 `row_max` 保持 `[M, 1]` 形状，方便与原始 `[M, N]` 的 `x` 做广播相减。


### 为什么减去最大值不会改变 Softmax 结果

可以先用直觉理解：Softmax 关心的是同一行里数字之间的相对大小，而不是它们整体加了多少。

例如：

```python
[10, 11, 12]
```

每个数字都减去 12 后变成：

```python
[-2, -1, 0]
```

最大的位置仍然是原来的最大位置，数字之间的差距也没有变。这样做能让 `exp` 计算更稳定，因为输入不会那么大。


## 4. 计算流程拆解

行 Softmax 可以拆成五步：

| 步骤 | PyPTO 表达 | 作用 |
| --- | --- | --- |
| 1 | `pypto.amax(x, dim=-1, keepdim=True)` | 得到每一行最大值 |
| 2 | `x - row_max` | 提升数值稳定性 |
| 3 | `pypto.exp(shifted)` | 计算指数 |
| 4 | `pypto.sum(exp, dim=-1, keepdim=True)` | 得到每一行指数和 |
| 5 | `exp / esum` | 归一化 |

通过这个拆解可以看到，Softmax 并不是单个神秘操作，而是规约和逐元素计算的组合。


## 4.1 从前面的问题回到 Softmax

本章前面的几个问题都可以在 Softmax 中串起来：

1. `amax(x, dim=-1, keepdim=True)` 是 reduction，用于得到每一行最大值。
2. `x - row_max` 是 elementwise，同时依赖广播，因为 `row_max.shape` 是 `[batch, 1]`。
3. `exp(shifted)` 是 elementwise，每个位置独立取指数。
4. `sum(exp, dim=-1, keepdim=True)` 是 reduction，用于得到每一行指数和。
5. `exp / row_sum` 是 elementwise，同时依赖广播。

Softmax 的输出满足每一行加起来约等于 1：

```text
softmax_i = exp_i / sum(exp)
sum(softmax_i) = sum(exp_i / sum(exp)) = 1
```

因此，Softmax 是本章概念的组合练习：shape、broadcast、elementwise、reduction、keepdim 都会同时出现。

## 4.2 与前面基础算子的关系

行 Softmax 正好把本章前面学过的内容组合起来：

| Softmax 步骤 | 计算类型 | 关键点 |
| --- | --- | --- |
| `amax(x, dim=-1, keepdim=True)` | reduction | 按行取最大值，保留 `[8, 1]` 形状 |
| `x - row_max` | elementwise | 依赖广播，提升数值稳定性 |
| `exp(shifted)` | elementwise | 每个位置独立取指数 |
| `sum(exp, dim=-1, keepdim=True)` | reduction | 按行求指数和，保留 `[8, 1]` 形状 |
| `exp / esum` | elementwise | 依赖广播完成归一化 |
| `out.move(...)` | 输出写回 | 把最终表达式写入 host 侧传入的输出 Tensor |

因此，章节实践不是新 API 的堆叠，而是把逐元素、规约、广播和输出写回组织成一个完整算子。


## 5. 编写 PyPTO Softmax 算子

下面将上述五步写入一个 PyPTO JIT 函数。


## 5.1 Softmax 算子规格

| 项目 | 说明 |
| --- | --- |
| 输入 `x` | FP32 Tensor，示例 shape 为 `[8, 8]` |
| 输出 `out` | FP32 Tensor，shape 为 `[8, 8]` |
| 规约维度 | `dim=-1`，按每一行内部计算 |
| 中间变量 `row_max` | shape 为 `[8, 1]`，用于数值稳定 |
| 中间变量 `exp` | shape 为 `[8, 8]` |
| 中间变量 `esum` | shape 为 `[8, 1]`，用于归一化 |
| 验证参考 | `torch.softmax(x, dim=-1)` |

这个规格表把 Softmax 拆成了可验证的中间步骤，后面阅读代码时会更清晰。


In [ ]:
import os

# 环境准备：参见 01.01_chapter_intro.ipynb §1 的详细说明
# ASCEND_RT_VISIBLE_DEVICES 必须在 import torch 之前设置才生效。
# 如需指定其他 NPU，请在启动 Notebook 前设置 ASCEND_RT_VISIBLE_DEVICES。
# os.environ.setdefault("ASCEND_RT_VISIBLE_DEVICES", "13")

import torch
import pypto

try:
    import torch_npu  # noqa: F401
except Exception:
    torch_npu = None


def reset_pypto_notebook_state():
    "Clean stale PyPTO recording state before defining a JIT kernel."
    try:
        pypto.reset()
    except Exception:
        pass

    try:
        from pypto._controller import Controller
        Controller.end_function()
    except Exception:
        pass


def get_device():
    if torch_npu is None:
        return "cpu"

    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    try:
        torch.npu.set_device(device_id)
    except Exception as exc:
        print("NPU device is not ready:", exc)
        return "cpu"
    return f"npu:{device_id}"


reset_pypto_notebook_state()
device = get_device()
RUN_MODE = pypto.RunMode.NPU if device != "cpu" else pypto.RunMode.SIM

# print("ASCEND_RT_VISIBLE_DEVICES:", os.environ.get("ASCEND_RT_VISIBLE_DEVICES", "<未设置>"))
print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<未设置，默认 0>"))
print("device:", device)

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def row_softmax_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    row_max = pypto.amax(x, dim=-1, keepdim=True)
    shifted = x - row_max
    exp = pypto.exp(shifted)
    esum = pypto.sum(exp, dim=-1, keepdim=True)
    out.move(exp / esum)


def main_row_softmax():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    x = torch.randn((8, 8), dtype=torch.float32, device=device)
    out = torch.empty_like(x)

    row_softmax_kernel(x, out)
    ref = torch.softmax(x, dim=-1)

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-3, atol=1e-3)
    print("row_softmax_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("输入 shape:", tuple(x.shape), "输出 shape:", tuple(out.shape))
    print("最大误差:", max_diff)
    print("输出前 2x4:", out[:2, :4].cpu())
    print("输出每行求和:", out.sum(dim=-1).cpu())


main_row_softmax()


`row_softmax_kernel` 按稳定版 Softmax 的顺序组织计算：先沿最后一维取每行最大值，再做平移、指数、按行求和，最后用指数值除以本行总和写回 `out`。这里的 `keepdim=True` 是为了让 `[batch, 1]` 形式的中间结果可以自然广播回原始 shape。


### 代码细节解释

- `pypto.amax(x, dim=-1, keepdim=True)`：得到每一行最大值，shape 从 `[8, 8]` 变为 `[8, 1]`。
- `shifted = x - row_max`：利用广播完成逐元素相减。
- `pypto.exp(shifted)`：对每个元素计算指数。
- `pypto.sum(exp, dim=-1, keepdim=True)`：得到每一行指数和，shape 为 `[8, 1]`。
- `out.move(exp / esum)`：利用广播完成归一化，并写回输出 Tensor。

这里最关键的两个点是 `dim=-1` 和 `keepdim=True`。前者决定沿最后一个维度规约，后者保证后续广播形状清晰。


### 把 Softmax Kernel 解释为自然语言

这段代码可以解释为五步：

1. 找到每一行最大的数字。
2. 每个数字都减去自己这一行的最大值。
3. 对减完后的每个数字做指数计算。
4. 把每一行的指数结果加起来。
5. 每个指数结果除以自己这一行的总和。

最后得到的 `out` 仍然和输入 `x` 一样大，但每一行都被转换成了“比例分布”。


这段代码中出现了两个规约：

- `pypto.amax`：按行取最大值。
- `pypto.sum`：按行求指数和。

也出现了三个逐元素操作：

- `x - row_max`
- `pypto.exp(shifted)`
- `exp / esum`

这正好体现了复杂算子由基础算子组合而成的思想。


上面的完整单元用来验证 `row_softmax_kernel`。它先清理 PyPTO Notebook 状态，再定义 JIT kernel；随后构造输入、提前分配输出 Tensor、调用 PyPTO kernel，并用 `torch.softmax(x, dim=-1)` 写出参考结果。打印每行求和是为了确认 Softmax 的输出确实接近概率分布。


**预期输出说明**

运行成功后，会看到 `row_softmax_kernel 验证通过`，并打印输出前几项和每行求和结果。Softmax 每行求和应接近 1。


## 6. 课后实践

练习目标是补全一个 Softmax 算子。实现时按照以下步骤逐步完成：

1. 计算每一行最大值。
2. 输入减去每一行最大值。
3. 计算指数。
4. 计算每一行指数和。
5. 写回 `exp / esum`。

核心语句：

```python
row_max = pypto.amax(x, dim=-1, keepdim=True)
exp = pypto.exp(x - row_max)
esum = pypto.sum(exp, dim=-1, keepdim=True)
out.move(exp / esum)
```


## 7. 章节自测

第 1 章自测题：

1. `set_vec_tile_shapes` 主要用于哪类计算？
2. `set_cube_tile_shapes` 主要用于哪类计算？
3. `dim=-1` 表示最后一行还是最后一个维度？
4. Softmax 中为什么要先减去每一行最大值？
5. `keepdim=True` 在 Softmax 中有什么作用？
6. `out.move(...)` 表示什么？

核心语句：

1. 主要用于逐元素、规约等向量类计算。
2. 主要用于矩阵乘法等 Cube 类计算。
3. 表示最后一个维度。
4. 为了提升数值稳定性，降低指数计算溢出风险。
5. 保留规约维度，方便后续广播计算。
6. 将计算表达的最终结果写回输出 Tensor。


## 8. 可运行章节练习：再次实现 Softmax Kernel

下面给出可直接验证的完整实现。它和主示例使用同一套数学流程，重点是从章节复盘角度再次阅读每一步：规约、广播、逐元素指数、规约求和、逐元素除法。


In [ ]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def row_softmax_practice_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    row_max = pypto.amax(x, dim=-1, keepdim=True)
    shifted = x - row_max
    exp = pypto.exp(shifted)
    esum = pypto.sum(exp, dim=-1, keepdim=True)
    out.move(exp / esum)


def main_row_softmax_practice():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    x = torch.randn((8, 8), dtype=torch.float32, device=device)
    out = torch.empty_like(x)

    row_softmax_practice_kernel(x, out)
    ref = torch.softmax(x, dim=-1)

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-3, atol=1e-3)
    print("row_softmax_practice_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("输出 shape:", tuple(out.shape), "最大误差:", max_diff)
    print("输出前 2x4:", out[:2, :4].cpu())
    print("输出每行求和:", out.sum(dim=-1).cpu())


main_row_softmax_practice()


`row_softmax_practice_kernel` 把 Softmax 的五个步骤完整写出。两次规约都使用 `keepdim=True`，因此 `row_max` 和 `esum` 都是 `[8, 1]`，可以自然广播回 `[8, 8]`。


验证逻辑仍然使用 `torch.softmax(x, dim=-1)` 作为参考结果，并打印输出每行求和。每一行求和接近 1，说明归一化结果符合 Softmax 的基本性质。


**预期输出说明**

运行成功后，会看到 `row_softmax_practice_kernel 验证通过`，并打印输出 shape、最大误差、输出前几项和每行求和结果。


核心语句：

```python
row_max = pypto.amax(x, dim=-1, keepdim=True)
shifted = x - row_max
exp = pypto.exp(shifted)
esum = pypto.sum(exp, dim=-1, keepdim=True)
out.move(exp / esum)
```

答案说明：

- 先减去 `row_max` 是为了数值稳定。
- 两次规约都使用 `keepdim=True`，是为了后续广播。
- 最后一行 `exp / esum` 让每一行归一化为概率分布。


## 9. 本章小结

通过第 1 章学习，已经建立 PyPTO 初级计算算子的基本开发方法：

- 用 JIT 函数描述可编译的 Tensor 计算。
- 用向量 Tile 和 Cube Tile 区分不同计算类型。
- 用规约和广播完成归一化类计算。
- 用切片和转置组织 Tensor 形状。
- 用 PyTorch 参考实现建立验证闭环。

后续学习自定义复杂算子时，可以沿用本章的方法：先拆解数学流程，再用 PyPTO 基础操作表达，最后进行结果验证。
